# SeCap MARS ground-truth video-ReID demo

This is a labelled cross-camera person video-ReID benchmark demonstration. MARS tracklets provide the ground-truth identity and camera label. The output videos show each model's hardest selected query, its ranked gallery candidates, the true rank, correct/incorrect result, and live benchmark statistics.

In [ ]:
import base64, hashlib, io, tarfile
from pathlib import Path
payload = base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest() == '95454d4b72f662675e05150f460fa7e273f92fb22e2356c4ff869624c31cf6cb'
repo = Path('/tmp/secap-mars'); repo.mkdir(exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(payload)) as bundle:
    bundle.extractall(repo, filter='data')
ns = {'__name__': 'secap_mars_runtime'}
exec(compile((repo/'kaggle/runtime.py').read_text(), 'runtime.py', 'exec'), ns)
ns['REPO'] = repo
ns['ART'] = Path('/kaggle/working/secap-mars-runtime'); ns['ART'].mkdir(exist_ok=True)
py = ns['install_environment']()
art = next(Path('/kaggle/input').rglob('checkpoints/CARGO/model_best.pth')).parents[2]
mars = next(Path('/kaggle/input').rglob('bbox_test'))
out = Path('/kaggle/working/secap_mars_video_reid'); out.mkdir(exist_ok=True)
ns['run']([py, 'video_demo/run_mars_reid_demo.py',
  '--data', mars.parent, '--output-dir', out,
  '--cargo-config', art/'CARGO/config.yaml', '--cargo-checkpoint', art/'checkpoints/CARGO/model_best.pth',
  '--lagper-config', art/'LAGPeR/config.yaml', '--lagper-checkpoint', art/'checkpoints/LAGPeR/model_best.pth',
  '--identities', '60'], cwd=repo)
print('SECAP_MARS_OUTPUT', out)


## Outputs

`/kaggle/working/secap_mars_video_reid/` contains one annotated cross-camera video per checkpoint and `mars_reid_report.json`. The coloured crop border denotes the benchmark's supplied person bounding box; no detector is being evaluated here.